# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via the Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print("Dataset name: ", metadata.name)
print("Description: ", metadata.description)
print("Version: ", getattr(metadata, 'version', ''))
print("License: ", getattr(metadata, 'license', ''))

## 2. Data Overview
Review available record sets, their `@id`s, and field information. All references to record sets, fields, and columns use their `@id` for accuracy and reproducibility.

In [ ]:
def get_record_sets(ds):
    """
    Collect available record sets from the dataset, with their @id and field info.
    """
    if not hasattr(ds.metadata, 'record_set'):
        print("No record sets available in this dataset.")
        return []
    rs_list = ds.metadata.record_set
    if not rs_list:
        print("No record sets available in this dataset.")
        return []
    print(f"Available record sets (by @id):")
    for rs in rs_list:
        print(f"  - {rs.id}    (name: {rs.name})")
        if hasattr(rs, 'field') and rs.field:
            print("    Fields:")
            for fld in rs.field:
                print(f"       * {fld.id}   (name: {fld.name}, type: {fld.data_type if hasattr(fld, 'data_type') else 'unknown'})")
        print()
    return rs_list

# List dataset record sets and their fields (by @id)
record_sets = get_record_sets(dataset)

In [ ]:
# If record_sets exist, let's show a preview of one record from each
if record_sets:
    for rs in record_sets:
        print(f"First record from record set {rs.id}:\n")
        try:
            rec_iter = dataset.records(record_set=rs.id)
            rec = next(rec_iter)
            pprint.pprint(rec)
        except Exception as e:
            print(f"(Failed to load records from {rs.id}) - {e}")
        print("\n" + "-"*60 + "\n")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use record set and field `@id`s discovered above.

In [ ]:
# --- Replace with your actual record set @ids ---
# Gather all record set @ids
if not record_sets:
    # If no record_sets found, try reading from the dataset.metadata.record_set attribute directly
    print("No record sets found in metadata. Please check dataset availability.")
    record_set_ids = []
else:
    record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for record set '{rs_id}' with columns: {df.columns.tolist()}")
        else:
            print(f"No records found in record set '{rs_id}'")
    except Exception as e:
        print(f"Could not load records for record set '{rs_id}': {e}")

# Show head of main record set (if available)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    display_columns = dataframes[main_rs_id].columns if main_rs_id in dataframes else []
    print(f"Columns in main record set '{main_rs_id}': {list(display_columns)}\n")
    if main_rs_id in dataframes:
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. Please refer to field `@id`s, not column names.

In [ ]:
# For demo, we'll try to select a likely numeric field.
# Replace these IDs with valid ones from your dataset in practice. We'll attempt to autodetect a numeric field.
import numpy as np

selected_rs_id = record_set_ids[0] if record_set_ids else None
numeric_field_id = None
group_field_id = None

if selected_rs_id is not None and selected_rs_id in dataframes:
    df = dataframes[selected_rs_id]
    # Try to pick a numeric column
    for col in df.columns:
        # Check if convertable to float
        try:
            df[col].astype(float)
            numeric_field_id = col
            break
        except: pass
    # Try to find a group field that's not numeric
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    if numeric_field_id is not None:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (Mean Value):\n", filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean)/std
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No suitable record set with data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. Here, we plot the numeric field distribution and normalized values.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id is not None and selected_rs_id in dataframes and numeric_field_id is not None:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field_id].dropna(), ax=axs[0], kde=True)
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    if f"{numeric_field_id}_normalized" in df.columns:
        sns.histplot(df[f"{numeric_field_id}_normalized"].dropna(), ax=axs[1], kde=True)
        axs[1].set_title(f"Distribution of normalized {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(12, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No visualization possible: missing numeric field or record set.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, process, and visualize the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

- We programmatically detected record sets and their field `@id`s, ensuring robust and reproducible references.
- Data from record sets was loaded, filtered by a numeric field, normalized, grouped, and visualized for further exploration.

By leveraging the Croissant schema's structure and `@id` references, this workflow is transparent and robust against changes in dataset structure.